# **Import Library**


In [ ]:
#ignore warnings
import warnings
warnings.filterwarnings('ignore')

#standard Libraries
import pandas as pd
import numpy as np

#visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (10,8)

from collections import Counter

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

import xgboost as xgb
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn import metrics


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/cse498r/human activities/train.csv')  #zara's path
# test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/cse498r/human activities/test.csv')    #zara's path

#train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CSE498R/train.csv')   #Baker's Path
#test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CSE498R/test.csv')     #Baker's Path

#train = pd.read_csv('/content/drive/MyDrive/final dataset/train.csv')
#test = pd.read_csv('/content/drive/MyDrive/final dataset/test.csv')
train = pd.read_csv('E:/CSE498R/final dataseet/test.csv')
test = pd.read_csv('E:/CSE498R/final dataseet/train.csv')


**Understanding the data**


In [ ]:
#see what is present in the data
train.head()

In [ ]:
#see what is present in the data
train.tail()


In [ ]:
train.info()

In [ ]:
train.shape

In [ ]:
train.describe()

In [ ]:
train.dtypes

In [ ]:
# how many records are available for each subjects
train.subject.value_counts()

In [ ]:
#count how many unique value every feature have
train.nunique()

In [ ]:
train['Activity'].unique()

# **Data Preprocessing**

##Checking for Duplicates

In [ ]:
print('Number of duplicates in train : ',sum(train.duplicated()))
print('Number of duplicates in test  : ',sum(test.duplicated()))

##Checking for Missing Values

In [ ]:
print('Number of Missing values in train : ',train.isna().values.sum())
print('Number of Missing values in test  : ',test.isna().values.sum())

In [ ]:
train.isnull().sum()

##Checking for class imbalance¶

In [ ]:
train['Activity'].value_counts()

here we can see our dataset is balanced

In [ ]:
sns.countplot(x=train['Activity'],data=train,order =train['Activity'].value_counts().index)
plt.title('Distribution of Activity(target) classes')
plt.xticks(rotation=50)
plt.show()

# Label Encoding

In [ ]:
train.columns

In [ ]:
#assigning label encoder
le = LabelEncoder()
#label encoding target class "Activity"
train[['Activity']]= train[['Activity']].apply(le.fit_transform)

In [ ]:
print(train.shape)
train.head(10)

In [ ]:
train.tail(20)

# **Exploratory Data Analysis (EDA)**


In [ ]:
train.head()

In [ ]:
#Counting how much parameter does each feature has
pd.DataFrame.from_dict(Counter([col.split('-')[0].split('(')[0]
                                for col in train.columns]),orient = 'index').rename(columns={0:'count'}).sort_values('count',ascending = False)

##**Relation of variables**

tBodyAccMag-mean feature analysis

In [ ]:
facetGrid = sns.FacetGrid(train,hue='Activity',height=5,aspect=3)
facetGrid.map(sns.distplot,'tBodyAccMag-mean()',hist = False).add_legend()

plt.annotate("Static Activities", xy = (-.98, 8), xytext = (-.8, 16), arrowprops={'arrowstyle': '-', 'ls': 'dashed'})
plt.annotate("Static Activities", xy = (-.98, 13), xytext = (-.8, 16), arrowprops={'arrowstyle': '-', 'ls': 'dashed'})
plt.annotate("Static Activities", xy = (-.98, 16), xytext = (-.8, 16), arrowprops={'arrowstyle': '-', 'ls': 'dashed'})

plt.annotate("Dynamic Activities", xy=(-0.2,3.25), xytext=(0.1, 9),arrowprops={'arrowstyle': '-', 'ls': 'dashed'})
plt.annotate("Dynamic Activities", xy=(0.1,2.18), xytext=(0.1, 9),arrowprops={'arrowstyle': '-', 'ls': 'dashed'})
plt.annotate("Dynamic Activities", xy=(-0.01,2.15), xytext=(0.1, 9),arrowprops={'arrowstyle': '-', 'ls': 'dashed'})

In [ ]:
sns.boxplot(x='Activity',y='tBodyAccMag-mean()',data=train, showfliers=False)
plt.ylabel("Body Accelerationmagnitude mean")
plt.title('Boxplot of tBodyAccmag-mean() column across various actvities')
plt.axhline(y=-0.8,xmin=0.01,dashes=(3,3))
plt.show()

# Data splitting and Feature scealing

In [ ]:
#Splitting the dataset
y = train['Activity']
X = train.drop(['Activity'], axis = 'columns')         #test data (removing target column)

In [ ]:
y.head(10)

In [ ]:
X.head(10)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 20, stratify = y)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
train.columns

In [ ]:
X_train.head()

In [ ]:
scaler = MinMaxScaler().fit(X_train)

In [ ]:
print(scaler)

In [ ]:
scaler.data_min_

In [ ]:
scaler.data_max_

In [ ]:
X_train.describe()

In [ ]:
scaler.feature_range

In [ ]:
scaler.transform(X_train)

In [ ]:
scaler = MinMaxScaler().fit(X_test)

In [ ]:
print(scaler)

In [ ]:
scaler.data_min_

In [ ]:
scaler.data_max_

In [ ]:
scaler.feature_range

In [ ]:
scaler.transform(X_test)

# Models Apply

# AdaBoost

In [ ]:
# Load libraries
from sklearn.ensemble import AdaBoostClassifier

# Create adaboost classifer object
abc = AdaBoostClassifier()

# Train Adaboost Classifer
model = abc.fit(X_train, y_train)

#Predict the response for test dataset
y_pred = model.predict(X_test)

# Model Accuracy, how often is the classifier correct?
print("Classification Report:\n", classification_report(y_test, y_pred))

print("\nTrain Accuracy: ", accuracy_score(y_train, abc.predict(X_train)))
print("Test Accuracy: ", accuracy_score(y_test, y_pred))


In [ ]:
# Import Support Vector Classifier
from sklearn.svm import SVC

svc=SVC(probability=True, kernel='linear')

# Create adaboost classifer object
abc =AdaBoostClassifier(n_estimators=50, base_estimator=svc,learning_rate=1,  random_state=42)

# Train Adaboost Classifer
model = abc.fit(X_train, y_train)

#Predict the response for test dataset
y_pred = model.predict(X_test)


# Model Accuracy, how often is the classifier correct?
print("Classification Report:\n", classification_report(y_test, y_pred))
print("\nTrain Accuracy: ", metrics.accuracy_score(y_train, abc.predict(X_train)))
print("Test Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
#Confusion Matrix

result = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", result)
print("\n\n")
# Calculate confusion matrix
confusion_mat = confusion_matrix(y_test, y_pred)

# Define class labels
class_names = ["Class 1", "Class 2", "Class 3", "Class 4", "Class 5", "Class 6"]

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Define the base estimator (e.g., Support Vector Machine)
base_estimator = SVC(probability=True, kernel='linear')

# Create the AdaBoostClassifier
adaboost_classifier = AdaBoostClassifier(base_estimator=base_estimator)

# Define the hyperparameters and their values to search over (reduced search space)
param_grid = {
    'n_estimators': [50, 200, 500],
    'learning_rate': [0.01, 0.1, 1.0],
}

# Create a GridSearchCV object with a smaller grid
grid_search_ada = GridSearchCV(adaboost_classifier, param_grid=param_grid, cv=3, verbose=2, n_jobs=-1)

# Fit the grid search to your data
grid_search_ada.fit(X_train, y_train)

# Get the best hyperparameters and the best estimator
best_params = grid_search_ada.best_params_
best_estimator = grid_search_ada.best_estimator_

# Print the best hyperparameters
print("Best Hyperparameters: ", best_params)
print("Best Estimator: ", best_estimator)

# Use the best model for prediction
best_classifier = grid_search_ada.best_estimator_
y_pred_best = best_classifier.predict(X_test)

# Evaluate the best model
accuracy_best = accuracy_score(y_test, y_pred_best)
print("Best Model Accuracy (Adaboost): {:.2f}%".format(accuracy_best * 100))
print("\n\nClassification Report:\n", classification_report(y_test, y_pred_best))

In [ ]:
# Create the LIME explainer
explainer = lime.lime_tabular.LimeTabularExplainer(training_data=np.array(X_train), mode="classification",feature_names= X_train.columns, categorical_features=[0])

# Choose a specific instance from the test set for explanation
explanation_instance = X_test.iloc[0]

# Get the explanation for the instance
explanation = explainer.explain_instance(explanation_instance, best_grid.predict_proba)

# Print the explanation for the predicted class
explanation.show_in_notebook()


In [ ]:
explanation.as_list()